# Kayak Recommandation de Destinations Françaises

> *Pipeline Data Engineering complet : API Météo · Scraping Booking.com · AWS S3 · AWS RDS*

**Périmètre :** 35 villes touristiques françaises
**Stack :** OpenWeatherMap · BeautifulSoup · AWS S3 · AWS RDS · Plotly

## 1. Imports & Configuration

In [14]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import re
import warnings
import requests
import time
warnings.filterwarnings('ignore')

print(" Imports OK")


 Imports OK


## 2. Partie 1 — Données Météo (OpenWeatherMap + Nominatim)

In [2]:
# Chargement des données météo collectées via OpenWeatherMap
# (voir kayak_part1_weather.py pour le code de collecte)

df_weather = pd.read_csv('../data/weather_france_cities.csv')
df_daily   = pd.read_csv('../data/weather_daily_details.csv')

print(f"Météo chargée : {len(df_weather)} villes, {len(df_daily)} observations journalières")
print(f"Colonnes météo : {df_weather.columns.tolist()}")
df_weather.head()


Météo chargée : 35 villes, 175 observations journalières
Colonnes météo : ['city_id', 'city_name', 'latitude', 'longitude', 'weather_score', 'temp', 'precip_prob']


,city_id,city_name,latitude,longitude,weather_score,temp,precip_prob
0,1,Mont Saint Michel,48.635954,-1.511460,67.350,13.446,19.0
1,2,St Malo,48.649518,-2.026041,67.570,13.294,18.0
2,3,Bayeux,49.276462,-0.702474,67.214,13.502,21.0
3,4,Le Havre,49.493898,0.107973,58.770,11.422,35.8
4,5,Rouen,49.440459,1.093966,63.248,13.792,40.0


In [3]:
# Score météo composite sur 7 jours par ville
# (calculé dans la collecte : température, probabilité de pluie, humidité)

df_weather_sorted = df_weather.sort_values('weather_score', ascending=False).reset_index(drop=True)

print("Top 10 destinations par score météo :")
print(df_weather_sorted[['city_name','weather_score','temp','precip_prob']].head(10).to_string(index=False))

TOP5 = df_weather_sorted.head(5)['city_name'].tolist()
print(f"\nTop 5 destinations : {TOP5}")


Top 10 destinations par score météo :
      city_name  weather_score   temp  precip_prob
      Collioure         85.932 17.556          0.0
          Nimes         84.360 16.580          0.0
        Bayonne         82.976 16.536          0.0
        Avignon         82.152 15.834          0.0
           Uzes         81.562 15.804          0.0
Aix en Provence         81.496 14.662          0.0
       Biarritz         81.278 15.848          0.0
       Toulouse         79.308 14.774          0.0
    Carcassonne         78.560 14.938          0.0
      Montauban         78.350 14.894          0.0

Top 5 destinations : ['Collioure', 'Nimes', 'Bayonne', 'Avignon', 'Uzes']


In [4]:
# Visualisation : évolution météo journalière pour le Top 5

top5_daily = df_daily[df_daily['city'].isin(TOP5)].copy()

fig_daily = px.line(
    top5_daily, x='date', y='weather_score', color='city',
    markers=True,
    title="Évolution du Score Météo sur 7 jours — Top 5 Destinations",
    labels={'weather_score': 'Score météo', 'date': 'Date', 'city': 'Ville'}
)
fig_daily.update_layout(height=450, hovermode='x unified')
fig_daily.show()


In [5]:
# Carte des 35 destinations

fig_dest = px.scatter_mapbox(
    df_weather_sorted,
    lat='latitude', lon='longitude',
    size='weather_score',
    color='weather_score',
    color_continuous_scale='RdYlGn',
    hover_name='city_name',
    hover_data={'weather_score': ':.1f', 'temp': ':.1f', 'precip_prob': ':.0f',
                'latitude': False, 'longitude': False},
    zoom=4.5,
    center={'lat': 46.5, 'lon': 2.5},
    mapbox_style='carto-positron',
    title='Score Météo — 35 Destinations Françaises',
    size_max=28,
    labels={'weather_score': 'Score météo', 'temp': 'Temp (°C)', 'precip_prob': 'Pluie (%)'}
)

top5_df = df_weather_sorted[df_weather_sorted['city_name'].isin(TOP5)]
fig_dest.add_trace(go.Scattermapbox(
    lat=top5_df['latitude'], lon=top5_df['longitude'],
    mode='markers+text',
    marker=dict(size=20, color='gold', symbol='star'),
    text=[' ' + c for c in top5_df['city_name']],
    textposition='top right',
    name='Top 5',
    textfont=dict(size=10, color='black')
))

fig_dest.update_layout(height=600, margin=dict(t=50, b=0, l=0, r=0))
fig_dest.show()


## 3. Partie 2 Scraping Hôtels Booking.com

In [6]:
# Chargement des hôtels via Google Places API
# (voir kayak_part2_hotels_googleplaces.py — remplace le scraping Booking.com,
#  bloqué par leur protection anti-bot AWS WAF)

df_hotels_raw = pd.read_csv('../data/hotels.csv')

print(f"Hôtels bruts : {len(df_hotels_raw)}")
print(df_hotels_raw.head(3))

Hôtels bruts : 699
           city_name          hotel_name  \
0  Mont Saint Michel  Hôtel La Confiance   
1  Mont Saint Michel          Hôtel Vert   
2  Mont Saint Michel   Hôtel du Guesclin   

                                             address  rating  rating_count  \
0  2 Escalier de la Poste, 50170 Le Mont-Saint-Mi...     3.4         658.0   
1  8 Rte du Mont Saint-Michel, 50170 Le Mont-Sain...     4.0        1397.0   
2     Grande Rue, 50170 Le Mont-Saint-Michel, France     4.3         921.0   

    latitude  longitude  
0  48.635178  -1.510426  
1  48.614730  -1.509355  
2  48.635633  -1.509833  


In [7]:
print(df_hotels_raw.columns.tolist())
print(df_hotels_raw.head(2))

['city_name', 'hotel_name', 'address', 'rating', 'rating_count', 'latitude', 'longitude']
           city_name          hotel_name  \
0  Mont Saint Michel  Hôtel La Confiance   
1  Mont Saint Michel          Hôtel Vert   

                                             address  rating  rating_count  \
0  2 Escalier de la Poste, 50170 Le Mont-Saint-Mi...     3.4         658.0   
1  8 Rte du Mont Saint-Michel, 50170 Le Mont-Sain...     4.0        1397.0   

    latitude  longitude  
0  48.635178  -1.510426  
1  48.614730  -1.509355  


In [8]:
# Score composite : note pondérée par le nombre d'avis
df_hotels = df_hotels_raw.dropna(subset=['hotel_name', 'rating', 'latitude', 'longitude']).copy()
df_hotels['score'] = df_hotels['rating'] * (df_hotels['rating_count'].clip(upper=500) ** 0.3)
df_hotels['hotel_id'] = range(1, len(df_hotels) + 1)

print(f"{len(df_hotels)} hôtels nettoyés avec scores")
print(df_hotels[['city_name', 'hotel_name', 'score']].head(10).to_string(index=False))

699 hôtels nettoyés avec scores
        city_name                                                                 hotel_name     score
Mont Saint Michel                                                         Hôtel La Confiance 21.936630
Mont Saint Michel                                                                 Hôtel Vert 25.807800
Mont Saint Michel                                                          Hôtel du Guesclin 27.743386
Mont Saint Michel                              B&B HOTEL Avranches Baie du Mont Saint-Michel 25.807800
Mont Saint Michel                                    Hôtel Gabriel - Hôtel Mont Saint Michel 25.807800
Mont Saint Michel                                   ibis Pontorson Baie du Mont-Saint-Michel 27.098191
Mont Saint Michel                                            Hôtel Mercure Mont-Saint-Michel 27.098191
Mont Saint Michel                                                           L'Etape du Mont, 21.794191
Mont Saint Michel The Originals Boutique,

In [10]:

# Top 20 hôtels (score pondéré note × avis)
top20_hotels = df_hotels.dropna(subset=['score', 'latitude', 'longitude']) \
                         .sort_values('score', ascending=False) \
                         .head(20)

print("Top 20 hôtels les mieux notés :")
print(top20_hotels[['hotel_name', 'city_name', 'rating', 'rating_count', 'score']].to_string(index=False))

Top 20 hôtels les mieux notés :
                                    hotel_name                    city_name  rating  rating_count     score
                             Hôtel Casa Marina     Saintes Maries de la mer     4.8        1232.0 30.969361
                                  Pullman Lyon                         Lyon     4.8        3146.0 30.969361
                    Hôtel Pêche de Vigne & Spa Chateau du Haut Koenigsbourg     4.8         504.0 30.969361
                            Hôtel Le Colombier             Gorges du Verdon     4.8         528.0 30.969361
    Aparthotel Adagio Toulouse Centre La Grave                     Toulouse     4.8         550.0 30.969361
                                  Colmar Hotel                       Colmar     4.8        1918.0 30.969361
                        Hôtel Le Nouveau Monde                      St Malo     4.7        2659.0 30.324166
                        Hôtel De la Plage HDLP           Bormes les Mimosas     4.7         527.0 30.324

In [20]:
# Carte Top 20 Hôtels
fig_hotels = px.scatter_mapbox(
    top20_hotels,
    lat='latitude', lon='longitude',
    color='hotel_name',        # ← changé : affiche le nom de l'hôtel dans la légende
    size='score',
    size_max=15,
    hover_name='hotel_name',
    hover_data={'score': True, 'city_name': True, 'latitude': False, 'longitude': False},
    zoom=4.5,
    center={'lat': 46.5, 'lon': 2.5},
    mapbox_style='carto-positron',
    title='Top 20 Hôtels les Mieux Notés Meilleures Destinations',
    labels={'score': 'Score pondéré', 'hotel_name': 'Hôtel'}
)
fig_hotels.update_layout(height=600, margin=dict(t=50, b=0, l=0, r=0))
fig_hotels.show()

## 4. Partie 3 — ETL : AWS S3 → AWS RDS

In [15]:
# Dataset final consolidé (météo + hôtels) pour l'entrepôt de données
df_final = df_weather_sorted.merge(
    df_hotels[['city_name', 'hotel_name', 'score']],
    left_on='city_name', right_on='city_name', how='left'
)
print(f"Dataset final : {df_final.shape}")
df_final.head()


Dataset final : (699, 9)


,city_id,city_name,latitude,longitude,weather_score,temp,precip_prob,hotel_name,score
0,28,Collioure,42.52505,3.083155,85.932,17.556,0.0,Hôtel-Restaurant Les Roches Brunes,27.409324
1,28,Collioure,42.52505,3.083155,85.932,17.556,0.0,Hôtel Princes de Catalogne,26.452995
2,28,Collioure,42.52505,3.083155,85.932,17.556,0.0,Le Relais des Trois Mas,27.010354
3,28,Collioure,42.52505,3.083155,85.932,17.556,0.0,Hôtel La Casa Païral,29.661151
4,28,Collioure,42.52505,3.083155,85.932,17.556,0.0,Hôtel La Frégate,26.452995


In [ ]:
# ── Chargement sur AWS S3 (Data Lake) ──────────────────────────────────
# (voir kayak_part3_etl.py pour le code complet)
#
# Structure S3 :
#   s3://your-bucket/raw/weather_france_cities.csv
#   s3://your-bucket/raw/hotels_france.csv
#   s3://your-bucket/processed/final_kayak_data.csv

import boto3, io
from dotenv import load_dotenv
import os

load_dotenv()

try:
    s3 = boto3.client(
        's3',
        aws_access_key_id     = os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY'),
        region_name           = os.getenv('AWS_DEFAULT_REGION', 'eu-west-3')
    )
    BUCKET = os.getenv('S3_BUCKET_NAME')

    def upload_to_s3(df, key):
        buf = io.StringIO()
        df.to_csv(buf, index=False)
        s3.put_object(Bucket=BUCKET, Key=key, Body=buf.getvalue())
        print(f"   s3://{BUCKET}/{key} ({len(df)} lignes)")

    print(" Upload vers S3...")
    upload_to_s3(df_weather,  "raw/weather_france_cities.csv")
    upload_to_s3(df_hotels,   "raw/hotels_france.csv")
    upload_to_s3(df_daily,    "raw/weather_daily_details.csv")
    upload_to_s3(df_final,    "processed/final_kayak_data.csv")
    print("\n Data Lake S3 alimenté !")

except Exception as e:
    print(f" AWS non configuré : {e}")
    print("→ Configurer les variables dans .env pour activer S3")


 Upload vers S3...
   s3://marty95/raw/weather_france_cities.csv (35 lignes)
   s3://marty95/raw/hotels_france.csv (699 lignes)
   s3://marty95/raw/weather_daily_details.csv (175 lignes)
   s3://marty95/processed/final_kayak_data.csv (699 lignes)

 Data Lake S3 alimenté !


In [18]:
# ── ETL S3 - RDS PostgreSQL (Data Warehouse) ───────────────────────────────
from sqlalchemy import create_engine

RDS_HOST = os.getenv('RDS_HOST')
RDS_PORT = os.getenv('RDS_PORT', '5432')
RDS_DB   = os.getenv('RDS_DB')
RDS_USER = os.getenv('RDS_USER')
RDS_PASS = os.getenv('RDS_PASSWORD')

try:
    engine = create_engine(f"postgresql+psycopg2://{RDS_USER}:{RDS_PASS}@{RDS_HOST}:{RDS_PORT}/{RDS_DB}")

    df_weather.to_sql('cities_weather', engine, if_exists='replace', index=False)
    print(f"   Table 'cities_weather' : {len(df_weather)} lignes")

    df_hotels.to_sql('hotels', engine, if_exists='replace', index=False)
    print(f"   Table 'hotels' : {len(df_hotels)} lignes")

    df_daily.to_sql('weather_daily', engine, if_exists='replace', index=False)
    print(f"   Table 'weather_daily' : {len(df_daily)} lignes")

    q = pd.read_sql("SELECT city_name, weather_score, temp FROM cities_weather ORDER BY weather_score DESC LIMIT 5", engine)
    print("\n Top 5 depuis RDS :")
    print(q)

except Exception as e:
    print(f" RDS non configuré : {e}")
    print("→ Configurer les variables RDS dans .env pour activer PostgreSQL")


   Table 'cities_weather' : 35 lignes
   Table 'hotels' : 699 lignes
   Table 'weather_daily' : 175 lignes

 Top 5 depuis RDS :
   city_name  weather_score    temp
0  Collioure         85.932  17.556
1      Nimes         84.360  16.580
2    Bayonne         82.976  16.536
3    Avignon         82.152  15.834
4       Uzes         81.562  15.804


## 5. Conclusion

### Pipeline réalisé

```
Nominatim API          → Coordonnées GPS des 35 villes françaises
        ↓
OpenWeatherMap API     → Prévisions météo 7 jours / ville
        ↓
Score météo composite  → Classement des destinations (temp + pluie + humidité)
        ↓
Booking.com scraping   → Hôtels avec notes utilisateurs (BeautifulSoup)
        ↓
Géocodage Nominatim    → Coordonnées précises par hôtel (avec repli + jitter)
        ↓
AWS S3 (Data Lake)     → Stockage raw + processed (CSV)
        ↓
AWS RDS (PostgreSQL)   → Data Warehouse requêtable par l'équipe analytics
        ↓
Plotly Maps            → 2 cartes interactives livrées à l'équipe marketing
```

### Résultats

| Indicateur | Valeur |
|---|---|
| Villes analysées | 35 |
| Jours de prévisions | 7 |
| Hôtels scrapés | 25 |
| Top destination | Collioure (score : 85.9) |
| 2ème destination | Nîmes (score : 84.4) |
| 3ème destination | Bayonne (score : 83.0) |

### Recommandation Kayak

Le **Sud de la France** (Collioure, Nîmes, Avignon, Aix-en-Provence) offre les meilleures conditions météo. Les équipes marketing peuvent cibler ces destinations pour leurs campagnes estivales.